In [3]:
import os
from datetime import datetime

from trade_logger import TradeLogger
from trading_agent import TradingAgent

class WeeklyReport:
    """
    Generates a weekly performance report
    for the Bitcoin trading agent.
    """

    def __init__(self, trading_agent):

        self.agent = trading_agent
        self.logger = TradeLogger()

    # =========================================================
    # GENERATE REPORT
    # =========================================================

    def generate_report(self):

        portfolio = self.agent.get_portfolio()

        trades = self.logger.get_trades()

        total_trades = len(trades)

        buys = [
            trade
            for trade in trades
            if trade.get("action") == "BUY"
        ]

        sells = [
            trade
            for trade in trades
            if trade.get("action") == "SELL"
        ]

        total_fees = sum(
            float(trade.get("fee", 0))
            for trade in trades
        )

        total_buys = sum(
            float(trade.get("usd_amount", 0))
            for trade in buys
        )

        total_sells = sum(
            float(trade.get("usd_amount", 0))
            for trade in sells
        )

        return_pct = float(
            portfolio.get("return_pct", 0)
        )

        total_pnl = float(
            portfolio.get("total_pnl", 0)
        )

        report = f"""
BITCOIN TRADING AGENT
WEEKLY PERFORMANCE REPORT
========================================

Report Date:
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

PORTFOLIO
----------------------------------------
Portfolio Value:
${portfolio.get("portfolio_value", 0):,.2f}

Cash:
${portfolio.get("cash_usd", 0):,.2f}

BTC Quantity:
{portfolio.get("btc_quantity", 0):.8f}

BTC Price:
${portfolio.get("btc_price", 0):,.2f}

Total P&L:
${total_pnl:,.2f}

Return:
{return_pct:.2f}%

TRADING ACTIVITY
----------------------------------------
Total Trades:
{total_trades}

BUY Trades:
{len(buys)}

SELL Trades:
{len(sells)}

Total BUY Amount:
${total_buys:,.2f}

Total SELL Amount:
${total_sells:,.2f}

Total Fees:
${total_fees:,.2f}

SYSTEM
----------------------------------------
Paper Trading:
{self.agent.paper_trading}

Last Price:
${self.agent.last_price:,.2f}
"""

        return report

risk_manager.py created successfully!


In [18]:
from config_loader import (
    get_google_sheet_config,
    convert_config_values
)

rows = get_google_sheet_config()

config = convert_config_values(rows)

print(config)

{'budget_usd': 10000, 'dca_enabled': True, 'dca_drop_pct': 3, 'dca_amount': 500, 'dca_interval_hours': 24, 'atr_enabled': True, 'atr_period': 14, 'atr_multiplier': 1.5, 'max_position_pct': 30, 'portfolio_stop_pct': 25, 'trading_mode': 'hybrid', 'monitoring_interval_minutes': 30, 'paper_trading': True, 'llm_enabled': True, 'max_position_usd': 2000, 'max_trade_usd': 2000, 'global_stop_loss_pct': 50, 'max_active_trades': 20, 'trading_fee_pct': 0.0015, 'dca_buy_amount_usd': 500, 'llm_min_confidence': 0.6}


In [12]:
agent = TradingAgent(config)
result = agent.process_market_data(price=100000)
reporter = WeeklyReport(agent)

result = agent.process_market_data(
    price=100000,
    atr=1500,
    rsi=55,
    macd=100,
    macd_signal=80,
    volume_ratio=1.2
)

print("Hybrid:", result.get("hybrid_recommendation"))
print("Trade:", result.get("trade"))

report = reporter.generate_report()

print(report)

DCA BUY executed: $500.00 at $100,000.00
Hybrid: AUTO
Trade: None

BITCOIN TRADING AGENT
WEEKLY PERFORMANCE REPORT

Report Date:
2026-08-31 13:14:44

PORTFOLIO
----------------------------------------
Portfolio Value:
$9,999.25

Cash:
$9,499.25

BTC Quantity:
0.00500000

BTC Price:
$100,000.00

Total P&L:
$-0.75

Return:
-0.01%

TRADING ACTIVITY
----------------------------------------
Total Trades:
1

BUY Trades:
1

SELL Trades:
0

Total BUY Amount:
$500.00

Total SELL Amount:
$0.00

Total Fees:
$0.75

SYSTEM
----------------------------------------
Paper Trading:
True

Last Price:
$100,000.00



In [9]:
from pathlib import Path

log_file = Path("../logs/trades.csv")

log_file.write_text(
    "timestamp,action,strategy,btc_price,btc_quantity,usd_amount,fee,reason\n"
)

print("trades.csv has been cleared.")

trades.csv has been cleared.


In [13]:
import pandas as pd

df = pd.read_csv("../logs/trades.csv")
df

,timestamp,action,strategy,btc_price,btc_quantity,usd_amount,fee,reason
2026-08-31T13:14:35.433462,BUY,DCA,100000.0,0.005,500.0,0.75,NaN,Initial DCA purchase


In [19]:
agent = TradingAgent(config)
result = agent.process_market_data(price=100000)
print(agent.get_portfolio())

DCA BUY executed: $500.00 at $100,000.00
{'timestamp': '2026-08-31T13:20:21.961165', 'cash_usd': 9499.25, 'btc_quantity': 0.005, 'btc_price': 100000.0, 'btc_value': 500.0, 'portfolio_value': 9999.25, 'initial_value': 10000.0, 'total_pnl': -0.75, 'return_pct': -0.0075, 'realized_pnl': 0.0, 'unrealized_pnl': 0.0, 'active_positions': 1, 'total_trades': 1}


In [17]:
result = agent.process_market_data(
    price=100000,
    atr=1500,
    rsi=55,
    macd=100,
    macd_signal=80,
    volume_ratio=1.2
)

print("Hybrid:", result.get("hybrid_recommendation"))
print("LLM:", result.get("llm"))
print("ATR:", result.get("atr"))
print("Trade:", result.get("trade"))

Hybrid: AUTO
LLM: {'market_regime': 'bullish', 'recommendation': 'SWING', 'confidence': 0.6, 'reason': 'MACD is positive and above its signal line and volume is slightly elevated, indicating bullish momentum. RSI is neutral (~55) and ATR (1.5% of price) shows moderate volatility. Missing moving averages and recent return data reduce certainty.', 'suggested_atr_multiplier': 2.0}
ATR: {'action': 'BUY', 'price': 100000.0, 'atr': 1500.0, 'stop_price': 97750.0, 'atr_multiplier': 1.5, 'confirmations': 3, 'reason': 'RSI supports momentum; MACD bullish; volume breakout'}
Trade: None


In [20]:
result = agent.process_market_data(
    price=100000,
    atr=1500,
    rsi=55,
    macd=100,
    macd_signal=80,
    volume_ratio=1.2
)

print("Hybrid:", result.get("hybrid_recommendation"))
print("LLM:", result.get("llm"))
print("ATR:", result.get("atr"))
print("Trade:", result.get("trade"))

Hybrid: AUTO
LLM: {'market_regime': 'bullish', 'recommendation': 'SWING', 'confidence': 0.65, 'reason': 'MACD is above its signal line indicating bullish momentum, RSI is neutral-to-slightly-bullish (~55) supporting continuation, and volume is elevated (volume_ratio 1.2) confirming interest. ATR relative to price (~1.5%) is moderate, so this looks like a trending bullish environment rather than extreme volatility.', 'suggested_atr_multiplier': 1.5}
ATR: {'action': 'BUY', 'price': 100000.0, 'atr': 1500.0, 'stop_price': 97750.0, 'atr_multiplier': 1.5, 'confirmations': 3, 'reason': 'RSI supports momentum; MACD bullish; volume breakout'}
Trade: None
